## 1. Load Results

First, let's load the JSONL results file and explore its structure.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load results from JSONL file
results_file = "../aria_inference_20251208_190635.jsonl"  # Update with your actual filename

results = []
with open(results_file, 'r') as f:
    for line in f:
        results.append(json.loads(line))

df = pd.DataFrame(results)

print(f"Loaded {len(df)} results")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Display first few results
df.head()

In [ ]:
columns = [
    "track_id", 
    "num_stems", 
    "stem_names", 
    "num_ai_stems", 
    "ai_stem_indices", 
    "ai_codec", 
    "real_probability", 
    "encoder_distribution", 
    "most_likely_encoder", 
    "error",
]
# df[columns]

## 2. Basic Statistics

Let's get an overview of the dataset and predictions.

In [ ]:
# Remove any error rows
df_valid = df[df['prediction'] != 'error'].copy()
print(f"Valid results: {len(df_valid)} (removed {len(df) - len(df_valid)} errors)")

# Basic stats
print(f"\nUnique tracks: {df_valid['track_id'].nunique()}")
print(f"Unique codecs: {df_valid['ai_codec'].unique()}")
print(f"\nPrediction distribution:")
print(df_valid['prediction'].value_counts())

print(f"\nNumber of AI stems distribution:")
print(df_valid['num_ai_stems'].value_counts().sort_index())

In [ ]:
# Summary statistics for probabilities
print("Real Probability Statistics:")
print(df_valid['real_probability'].describe())

print("\nConfidence Statistics:")
print(df_valid['confidence'].describe())

## 3. Distribution Visualizations

Let's visualize how the model assigns probabilities to different types of mixes.

In [ ]:
# Distribution of real probabilities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df_valid['real_probability'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0.5, color='red', linestyle='--', label='Decision threshold')
axes[0].set_xlabel('Real Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Real Probabilities')
axes[0].legend()

# By number of AI stems
for num_ai in sorted(df_valid['num_ai_stems'].unique()):
    subset = df_valid[df_valid['num_ai_stems'] == num_ai]
    axes[1].hist(subset['real_probability'], bins=30, alpha=0.5, label=f'{num_ai} AI stems')

axes[1].axvline(0.5, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Real Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Real Probability by Number of AI Stems')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
df_valid[columns]

## 4. Ridgeline Plot (Joyplot)

Visualize the distribution of real probabilities for each number of AI stems using a ridgeline plot.

In [ ]:
# Install joypy if not already installed
# !pip install joypy

import joypy
from matplotlib import cm

# Prepare data for ridgeline plot
fig, axes = joypy.joyplot(
    df_valid, 
    column='real_probability',
    by='num_ai_stems',
    figsize=(12, 8),
    alpha=0.7,
    colormap=cm.viridis_r,
    linewidth=1.5,
    legend=True,
    grid=True,
    fade=True,
    range_style='all',
    # x_range=[0, 1],
)

# Customize
plt.xlabel('Real Probability', fontsize=12)
plt.suptitle('Distribution of Real Probabilities by Number of AI Stems', 
             fontsize=14, fontweight='bold', y=0.98)

# Add decision threshold line
for ax in axes:
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.7, zorder=0)

# Add labels for each ridgeline
for i, ax in enumerate(axes[:-1]):  # Skip the last axis (x-axis)
    num_ai = sorted(df_valid['num_ai_stems'].unique())[i]
    total_count = len(df_valid[df_valid['num_ai_stems'] == num_ai])
    real_count = len(df_valid[(df_valid['num_ai_stems'] == num_ai) & (df_valid['prediction'] == 'real')])
    fake_count = total_count - real_count
    
    # Add text annotation on the right side
    ax.text(1.02, 0.5, f'n={total_count}\n{num_ai} AI stems\nReal: {real_count}\nFake: {fake_count}', 
            transform=ax.transAxes, fontsize=9, va='center')

plt.tight_layout()
plt.show()